# GeoSR-4 — D040 ablation: isolate ICNR vs perceptual loss (Kaggle GPU)

D023 changed two things at once (ICNR upsample init + VGG perceptual loss) for time reasons, so we never learned which one actually caused the visible sharpness improvement (D029). These two runs each change only ONE variable from the D013 baseline (neither), so combined with D013 (neither) and D028 (both, already have those numbers) this gives a full 2x2 comparison.

**Before running, in the notebook's right-hand Settings panel:**
- Accelerator: **GPU P100** (simplest -- these runs don't strictly need `--amp`). If you'd rather use **GPU T4 x2**, that's fine too: the training commands below already include `--amp` so T4's Tensor Cores actually get used (otherwise T4 is slower than P100 for a plain-fp32 script, see D035). Either way, only one GPU actually gets used -- this script has no multi-GPU logic, so T4 x2 doesn't mean 2x speed.
- Internet: ON (needed for git clone + dataset download). **If you hit `Could not resolve host: github.com`**: this almost always means your Kaggle account isn't phone-verified yet (Settings → Account → Phone Verification) -- verify it, then fully restart the session (not just re-run cells) before trying again.

Kaggle gives ~30 GPU-hours/week on the free tier -- these two runs (20 epochs each, ~40 min each on a T4/P100 based on earlier SwinIR timing) take a small fraction of that, well within one session.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
Kaggle's default image already has PyTorch with CUDA -- only the packages it's missing get installed.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3a. ICNR only (no perceptual loss)
Compares against D013's plain baseline (neither ICNR nor perceptual) -- isolates ICNR's own contribution.

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 20 \
  --batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/swinir_icnr_only \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_icnr_only/swinir_epoch19.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 3b. Perceptual loss only (no ICNR -- random upsample init)
Also compares against D013's baseline -- isolates the perceptual loss's own contribution.

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 20 \
  --batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --no-icnr-init \
  --amp \
  --checkpoint-dir experiments/swinir_perceptual_only \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_perceptual_only/swinir_epoch19.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 4. Get the checkpoints out
Kaggle has no `google.colab.files.download` -- copy them to `/kaggle/working/`, then they're downloadable from the notebook's **Output** tab (right-hand panel) once you save a version.

In [ ]:
import shutil
shutil.copy("experiments/swinir_icnr_only/swinir_epoch19.pt", "/kaggle/working/swinir_icnr_only_epoch19.pt")
shutil.copy("experiments/swinir_perceptual_only/swinir_epoch19.pt", "/kaggle/working/swinir_perceptual_only_epoch19.pt")
print("copied to /kaggle/working/ -- visible in the Output tab once you save a version of this notebook")